# Video-only RVQ-VAE + TF-IDF

Notebook dùng duy nhất đặc trưng ViTPose của bài báo. Chạy `smoke` trước, sau đó đổi thành `full` để đánh giá đủ 9 fold.

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/kaggle/input/datasets/xndung/fidgety/data")
REPO_URL = "https://github.com/xn-dung/Fidgety.git"
SOURCE_DIR = Path("/kaggle/working/Fidgety")
UPLOADED_PROJECT_DIR = None
RUN_MODE = "smoke"
SEED = 42
OUTPUT_DIR = Path(f"/kaggle/working/video_rvq_{RUN_MODE}")

if RUN_MODE == "smoke":
    FOLDS = [1]
    EPOCHS = 5
    PATIENCE = 3
elif RUN_MODE == "full":
    FOLDS = list(range(1, 10))
    EPOCHS = 80
    PATIENCE = 10
else:
    raise ValueError("RUN_MODE must be smoke or full")

## Chuẩn bị source

Nếu chưa đưa thư mục `đồ án` lên repository, hãy upload nó thành một Kaggle Dataset và đặt `UPLOADED_PROJECT_DIR` ở cell trên thành đường dẫn tới thư mục đó.

In [ ]:
import shutil
import subprocess
import sys

if UPLOADED_PROJECT_DIR is not None:
    PROJECT_DIR = Path(UPLOADED_PROJECT_DIR)
else:
    if SOURCE_DIR.exists():
        subprocess.run(["git", "-C", str(SOURCE_DIR), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(SOURCE_DIR)], check=True)
    PROJECT_DIR = SOURCE_DIR / "đồ án"

if not (PROJECT_DIR / "run.py").is_file():
    raise FileNotFoundError(f"Không tìm thấy project tại {PROJECT_DIR}. Hãy push source lên GitHub hoặc đặt UPLOADED_PROJECT_DIR.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "requirements.txt")], check=True)
print("Project:", PROJECT_DIR)

In [ ]:
import torch

if not DATA_ROOT.is_dir() and (DATA_ROOT.parent / "data.zip").is_file():
    DATA_ROOT = DATA_ROOT.parent / "data.zip"
if not DATA_ROOT.exists():
    raise FileNotFoundError(DATA_ROOT)
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle Settings > Accelerator")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data:", DATA_ROOT)
print("GPU:", torch.cuda.get_device_name(0))
print("Folds:", FOLDS)
print("Epochs:", EPOCHS)

## Huấn luyện và đánh giá

In [ ]:
command = [
    sys.executable, "-u", str(PROJECT_DIR / "run.py"),
    "--data-path", str(DATA_ROOT),
    "--output-dir", str(OUTPUT_DIR),
    "--folds", ",".join(map(str, FOLDS)),
    "--seed", str(SEED),
    "--epochs", str(EPOCHS),
    "--patience", str(PATIENCE),
    "--batch-size", "64",
    "--patch-size", "10",
    "--hidden-dim", "256",
    "--latent-dim", "64",
    "--codebook-size", "64",
    "--quantizers", "2",
    "--ngram-order", "2"
]
with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    returncode = process.wait()
if returncode:
    raise RuntimeError(f"Training failed with code {returncode}")

## Kết quả so với baseline video của bài báo

In [ ]:
import pandas as pd
from IPython.display import display

fold_results = pd.read_csv(OUTPUT_DIR / "fold_results.csv")
summary = pd.read_csv(OUTPUT_DIR / "summary.csv")
display(fold_results.round(4))
display(summary.round(4))
if RUN_MODE == "smoke":
    print("Smoke mode chỉ kiểm tra pipeline, không dùng để kết luận so sánh với bài báo.")
else:
    print("Published VID balanced accuracy: 90.66%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot = summary.copy()
values = 100 * plot["balanced_accuracy_mean"].to_numpy()
errors = 100 * (plot["balanced_accuracy_ci_high"] - plot["balanced_accuracy_mean"]).to_numpy()
positions = np.arange(len(plot))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(positions, values, yerr=errors if np.isfinite(errors).all() else None, capsize=5)
ax.axhline(90.6606467, color="black", linestyle="--", label="Published video CNN: 90.66%")
ax.set_xticks(positions, plot["classifier"], rotation=15)
ax.set_ylabel("Balanced accuracy (%)")
ax.set_ylim(0, 100)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "comparison.png", dpi=180)
plt.show()

In [ ]:
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print("Artifacts:", archive)